# Lab 3: A Hands-On Introduction to CNNs with PyTorch

Deep Learning, Semester 5, Plaksha University. Builds and trains a three-block convolutional network on the CIFAR-10 images (10 classes of 32 x 32 colour images).

## 1: Why Do We Need CNNs?

Imagine you want to classify a simple 100x100 pixel color image using a standard Multi-Layer Perceptron (MLP).

*   **Input Size:** 100 (width) * 100 (height) * 3 (RGB channels) = **30,000 input features**.
*   You would first have to *flatten* this image into a single giant vector of 30,000 numbers.

This approach has two major problems:

1.  **Huge Number of Parameters:** If your first hidden layer has just 500 neurons, you would have 30,000 x 500 = **15 million** weights for just that one layer! This is computationally expensive, slow to train, and very prone to **overfitting**.

2.  **Loss of Spatial Information:** When you flatten the image, you lose all the spatial relationships between pixels. A pixel's relationship with its immediate neighbors is lost. This is not how images work! In an image, local groups of pixels form meaningful patterns like edges, corners, and textures.

### The CNN Solution

CNNs are designed to solve these problems by using two main ideas:

1.  **Convolution:** Uses learnable filters (kernels) that slide over the image to detect local patterns (like edges, corners, textures). This preserves spatial information and drastically reduces parameters through **parameter sharing** (using the same filter across the whole image).

2.  **Pooling:** Down-samples the feature maps created by the convolution layers. This makes the representation smaller and more manageable, and also makes the network more robust to small shifts in the position of features.

## 2: Setting Up Our PyTorch Environment

First, let's import the libraries we need. We will be using `torch` for the core framework and `torchvision` for datasets, model architectures, and image transformations.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(0)

# Set device to GPU if available, otherwise CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 3: Loading and Preparing the CIFAR-10 Dataset

PyTorch's `torchvision` makes it very easy to load standard datasets. We will also apply some transformations to our images.

1.  **`transforms.ToTensor()`**: This converts the images from PIL Image format (or NumPy array) into PyTorch Tensors.
2.  **`transforms.Normalize()`**: This normalizes the tensor image with a mean and standard deviation.

In [ ]:
# Define a chain of transformations to apply to the images
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

# Set the batch size
batch_size = 64   # larger batches keep CPU training to a few minutes per epoch

# Download and load the training data
trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=0)

# Download and load the test data
testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=0)

# Define the class names
classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

### Let's Visualize Some Data

Let's check our data by plotting a few images from the training set.

In [ ]:
def imshow(img):
    # Un-normalize the image to display it correctly
    img = img / 2 + 0.5
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

# Get some random training images
dataiter = iter(trainloader)
images, labels = next(dataiter)

# Show images
imshow(torchvision.utils.make_grid(images[:8]))
# Print labels
print(' '.join(f'{classes[labels[j]]:5s}' for j in range(8)))

## 4: Building the CNN Model

In PyTorch, we define our neural network as a class that inherits from `nn.Module`.

- The layers of the network are defined in the `__init__` method.
- The forward pass (how data flows through the network) is defined in the `forward` method.

Our architecture will be:
`CONV1 -> RELU -> POOL1 -> CONV2 -> RELU -> POOL2 -> CONV3 -> RELU -> FC1 -> RELU -> FC2`

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)

        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)

        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):

        #Block 1
        x = self.pool(F.relu(self.conv1(x)))

        # Block 2
        x = self.pool(F.relu(self.conv2(x)))

        # Block 3
        x = self.pool(F.relu(self.conv3(x)))

        # Flatten
        x = torch.flatten(x, 1)  # Flatten all dimensions except batch

        # Fully connected layers
        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

# Create an instance of the network and move it to the correct device
net = Net().to(device)
print(net)

## 5: Defining a Loss Function and Optimizer

* **Loss Function**:  
  Which loss function should you use, and why?  
  For example, you can consider: `CrossEntropyLoss`, `BCELoss`, `MSELoss`, or `NLLLoss`.

* **Optimizer**:  
  Which optimizer would you choose to train the model, and why?  
  For example, you might use: `Adam`, `SGD with momentum`, or `RMSprop`.

**Answers**

- **Loss: `CrossEntropyLoss`.** This is a single-label, 10-class problem. Cross-entropy applies log-softmax to the raw scores (logits) and penalises a low probability on the true class. `BCELoss` is for binary or multi-label outputs, `MSELoss` is for regression, and `NLLLoss` would need an explicit log-softmax layer first.
- **Optimiser: `Adam` (learning rate 0.001).** It adapts the step size per parameter and combines momentum with RMS scaling, so it converges quickly with little tuning. SGD with momentum can generalise slightly better, but needs a carefully tuned learning rate schedule.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)

## 6: Training the Model

This is the core of the learning process. Unlike Keras's `.fit()` method, in PyTorch, we write the training loop ourselves. This gives us more control and is great for understanding what's happening under the hood.

The process for each batch is:
1.  Get a batch of `inputs` and `labels` from the `trainloader`.
2.  Move the data to the GPU .
3.  **Zero the gradients** This is critical!
4.  **Forward Pass**: Feed the inputs through the network.
5.  **Calculate Loss**: Compare the network's outputs with the true labels.
6.  **Backward Pass**: Calculate the gradients.
7.  **Update Weights**: The optimizer updates the network's weights based on the calculated gradients.

In [ ]:
num_epochs = 10
epoch_losses = []

for epoch in range(num_epochs):
    running_loss = 0.0
    epoch_total = 0.0

    for i, data in enumerate(trainloader, 0):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        # 1. Zero the parameter gradients
        optimizer.zero_grad()

        # 2. Forward pass
        outputs = net(inputs)

        # 3. Calculate loss
        loss = criterion(outputs, labels)

        # 4. Backward pass
        loss.backward()

        # 5. Optimize
        optimizer.step()

        running_loss += loss.item()
        epoch_total += loss.item()
        if (i + 1) % 200 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}], Loss: {running_loss / 200:.4f}')
            running_loss = 0.0

    epoch_losses.append(epoch_total / len(trainloader))

print('Finished Training')
plt.plot(range(1, num_epochs + 1), epoch_losses, marker='o')
plt.xlabel('Epoch'); plt.ylabel('Average training loss'); plt.title('Training loss per epoch'); plt.grid(alpha=0.3)
plt.savefig('images/lab03_training_loss.png')
plt.show()

## 7: Evaluating the Model

Now that the model is trained, let's see how it performs on the test dataset. We don't need to calculate gradients for evaluation, so we use `torch.no_grad()` to make the process faster and more memory-efficient.

We will also set our model to evaluation mode using `net.eval()`. This is important because some layers, like Dropout and BatchNorm, behave differently during training and evaluation.

In [ ]:
correct = 0
total = 0

net.eval()

with torch.no_grad():
    for data in testloader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)

        outputs = net(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy of the network on the 10000 test images: {accuracy:.2f} %')

## 8: Making Predictions on Individual Images

In [ ]:
# Get a batch of test images
dataiter = iter(testloader)
images, labels = next(dataiter)

# Print images and true labels
imshow(torchvision.utils.make_grid(images[:8]))
print('GroundTruth: ', ' '.join(f'{classes[labels[j]]:5s}' for j in range(8)))

# Get predictions
with torch.no_grad():
    outputs = net(images.to(device))
_, predicted = torch.max(outputs, 1)

print('Predicted:   ', ' '.join(f'{classes[predicted[j]]:5s}' for j in range(8)))

**Where to go from here?**
*   **Improve the model:** Try adding more layers, using different kernel sizes, or adding **Dropout** layers (`nn.Dropout`) to reduce overfitting.
*   **Data Augmentation:** Add more transformations (like random flips or rotations) to your `transform` pipeline to create a more robust model.

In [ ]:
# Per-class test accuracy
class_correct = {c: 0 for c in classes}
class_total = {c: 0 for c in classes}
net.eval()
with torch.no_grad():
    for images, labels in testloader:
        preds = net(images.to(device)).argmax(1).cpu()
        for label, pred in zip(labels, preds):
            class_total[classes[label]] += 1
            class_correct[classes[label]] += int(label == pred)
acc = {c: 100 * class_correct[c] / class_total[c] for c in classes}
for c in classes:
    print(f"{c:6s}: {acc[c]:.1f}%")
plt.figure(figsize=(8, 3.5))
plt.bar(acc.keys(), acc.values())
plt.ylabel("Test accuracy (%)"); plt.title("Accuracy per class")
plt.tight_layout()
plt.savefig("images/lab03_per_class.png")
plt.show()